# HADL – Odia QA data selection: full evaluation pipeline on Kaggle

Runs `evaluation_pipeline/run_pipeline.sh` from the **`main`** branch of `pratyushk22/hadl_courseproject`:
re-profile the time proxy on this GPU → costed pool → subset selection → Qwen3-1.7B + LoRA training → evaluation
→ covariance-shift diagnostic → tables → figures. Everything runs inside this notebook; nothing has to be run afterwards.

**One-time notebook settings (right-hand panel → *Settings*)**
1. **Accelerator:** `GPU T4 x2` (both GPUs are used in parallel, one training run per GPU).
2. **Internet:** `On` (needs a phone-verified Kaggle account). Used to download the code, Qwen3-1.7B, the e5 embedding
   model and the embedding files from the team Google Drive.
3. **Persistence:** `Files only` (lets an interrupted interactive session resume); **Environment:** `Pin to original environment`.
4. *(Optional, only to push results to GitHub)* **Add-ons → Secrets** → add `GITHUB_TOKEN`
   (a GitHub token with *Contents: read & write* on `pratyushk22/hadl_courseproject`) and tick it for this notebook.

**How to run:** **Save Version → Save & Run All (Commit)**; it keeps running with the browser closed (limit 12 h).
The default configuration took about 3 h on 2×T4 in our run. Results land in `/kaggle/working/outputs`
(notebook *Output* tab; everything except the costed pool is also zipped into `hadl_results.zip`).

In [ ]:
# ---------------- configuration ----------------
import os, subprocess, torch

REPO_URL = "https://github.com/pratyushk22/hadl_courseproject.git"
BRANCH   = "main"                    # branch the code is cloned from
PUSH_RESULTS_TO_BRANCH = False      # True -> commit outputs to RESULTS_BRANCH (needs the GITHUB_TOKEN secret)
RESULTS_BRANCH = "kaggle-results"   # a separate branch, so main is never changed automatically

RUN = {
    "PROFILE": "1",                                   # re-fit the time proxy on this GPU (recommended)
    "SEEDS": "42",                                    # "42 43 44" for error bars (3x GPU time)
    "BUDGET_X_REAL": "1.0",                           # budget = 1.0 x proxy time of the real train split
    "METHODS": "all",                                 # "all" methods used
    "REFERENCE": "real_only",                         # extra reference run; "" to skip
    "EPOCHS": "1",
}

# ---- hardware-dependent settings (auto) ----
n_gpu = torch.cuda.device_count()
assert n_gpu > 0, "No GPU: set Settings -> Accelerator -> GPU T4 x2"
gpu_names = [torch.cuda.get_device_name(i) for i in range(n_gpu)]
mem_gb = min(torch.cuda.get_device_properties(i).total_memory for i in range(n_gpu)) / 1024**3
small_gpu = mem_gb < 30                               # T4 / P100 = 16 GB
RUN.update({
    "GPUS": str(n_gpu),
    "BATCH": "2" if small_gpu else "4",               # same effective batch 16 either way
    "ACCUM": "8" if small_gpu else "4",
    "EVAL_BS": "4" if small_gpu else "8",
    "DTYPE": "auto",                                  # fp16 on T4/P100, bf16 on A100/L4
})
REPO_DIR = "/tmp/hadl_courseproject"                  # code (not saved as notebook output)
OUT_DIR  = "/kaggle/working/outputs"                  # results (saved as notebook output)
os.environ.update(RUN)
os.environ["HADL_OUTPUT_ROOT"] = OUT_DIR
os.environ["PYTHON"] = "python"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print(f"{n_gpu} GPU(s): {gpu_names}, {mem_gb:.0f} GB each")
print({k: v for k, v in RUN.items()})

In [ ]:
# ---------------- get the code ----------------
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
if not os.path.isfile(f"{REPO_DIR}/evaluation_pipeline/run_pipeline.sh"):
    raise SystemExit(f"'{BRANCH}' of {REPO_URL} does not contain evaluation_pipeline/ yet. "
                     "Merge the evaluation-pipeline branch first, or set REPO_URL/BRANCH to where it lives.")

In [ ]:
# ---------------- dependencies (versions tested with this pipeline) ----------------
!pip install -q "transformers==4.57.*" "peft==0.22.*" "accelerate>=0.34" "datasets>=2.19" "jinja2>=3.1" gdown pyarrow scikit-learn scipy matplotlib pytest
import transformers, peft
print("transformers", transformers.__version__, "| peft", peft.__version__, "| torch", torch.__version__)

In [ ]:
# ---------------- quick sanity check: unit tests (CPU, ~10 s) ----------------
!cd {REPO_DIR}/evaluation_pipeline && python -m pytest -q tests -p no:cacheprovider

In [ ]:
%%bash
# ---------------- run the whole pipeline (steps 0-7) ----------------
# Finished runs / evaluations are skipped, so re-running this cell resumes after an interruption.
# Logs: /kaggle/working/outputs/logs/ ({train,eval}_shard*.log, covariance_diagnostic.log)
set -eo pipefail
cd /tmp/hadl_courseproject/evaluation_pipeline
bash run_pipeline.sh 2>&1 | tee /kaggle/working/pipeline.log

In [ ]:
# ---------------- results ----------------
import json, glob
from IPython.display import Markdown, Image, display
R = f"{OUT_DIR}/results"
for f in ["results_table.md", "significance_vs_random.md", "covariance_diagnostic.md"]:
    p = f"{R}/{f}"
    if os.path.exists(p):
        display(Markdown(f"### {f}\n" + open(p).read()))
p = f"{R}/proxy_error_summary.json"
if os.path.exists(p):
    print("Time-proxy error:", json.dumps(json.load(open(p)), indent=1))
for png in ["qa_scores_test", "quality_vs_training_time_test", "training_time_predicted_vs_actual",
            "proxy_step_parity", "training_loss_curves", "pipeline_overhead"]:
    f = f"{OUT_DIR}/figures/{png}.png"
    if os.path.exists(f):
        display(Image(f, width=900))
for f in sorted(glob.glob(f"{OUT_DIR}/figures/selection_overview_b*.png")
                + glob.glob(f"{OUT_DIR}/figures/covariance_shift_b*.png")):
    display(Image(f, width=900))

In [ ]:
# ---------------- one zip with everything (also visible in the Output tab) ----------------
!cd /kaggle/working && zip -qr hadl_results.zip outputs -x "outputs/pool/*" && ls -lh hadl_results.zip

In [ ]:
# ---------------- optional: commit the results to RESULTS_BRANCH (never to main) ----------------
import shutil
if PUSH_RESULTS_TO_BRANCH:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    dst = f"{REPO_DIR}/evaluation_pipeline/outputs"
    for sub in ["selections", "runs", "eval", "results", "figures", "profiling", "logs"]:
        if os.path.isdir(f"{OUT_DIR}/{sub}"):
            shutil.rmtree(f"{dst}/{sub}", ignore_errors=True)   # replace the CPU-only selection outputs
            shutil.copytree(f"{OUT_DIR}/{sub}", f"{dst}/{sub}",
                            ignore=shutil.ignore_patterns("trainer_tmp", "*.pt"))
    git = lambda *a: subprocess.run(["git", "-C", REPO_DIR, *a], check=True)
    git("config", "user.name", "HADL Kaggle run")
    git("config", "user.email", "hadl-kaggle@users.noreply.github.com")
    git("add", "-A", "evaluation_pipeline/outputs")
    git("commit", "-m", f"Add GPU results from Kaggle ({', '.join(sorted(set(gpu_names)))})")
    push_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
    r = subprocess.run(["git", "-C", REPO_DIR, "push", push_url, f"HEAD:refs/heads/{RESULTS_BRANCH}"],
                       capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token, "***"))
    if r.returncode != 0:
        print(f"Push to {RESULTS_BRANCH} failed (branch exists with other commits, or the token lacks write access). "
              "Download hadl_results.zip instead.")
else:
    print("PUSH_RESULTS_TO_BRANCH is False - the results are in the Output tab (outputs/ and hadl_results.zip).")